# Save checksums as MD5

This notebook demoes the story RSPY-1242: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1242, in particular the system making sure we save checksums as MD5.

This demo proceeds in 5 steps:
1. Find an example item and retrieve its checksum
2. Retrieve and delete its checksum
3. Insert it in the catalog
4. Check the new checksum using the STAC API
5. Check the new checksum using the ODATA API

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
import pprint
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

## Stage an item in the catalog

In [ ]:
# Create a test collection 
CATALOG_COLLECTION_ID = "SPRINT42_1242_MD5_CHECKSUM"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

In [ ]:
# Stage an Auxip item in the new collection of the catalog
item_id = "S1A_AUX_PP2_V20200920T080000_G20200920T080000"
stac_filter = f"id in ('{item_id}')"
item_collection_auxip = auxip_client.search(
    method="GET",
    stac_filter=stac_filter
)

# Save the checksum and delete it from the product
original_checksum = item_collection_auxip.to_dict()["features"][0]["assets"]["S1A_AUX_PP2_V20200920T080000_G20200920T080000.SAFE"].pop("file:checksum")
original_checksum

In [ ]:
staged_item = stage_data(item_collection_auxip.to_dict(), items_id = [], catalog_collection_name = CATALOG_COLLECTION_ID)

In [ ]:
# Check the new checksum
original_catalog_item = staged_item[0].to_dict()

original_catalog_item["assets"]["S1A_AUX_PP2_V20200920T080000_G20200920T080000.SAFE"]["file:checksum"]
# d50110 is the multihash prefix for MD5 (https://multiformats.io/multihash/)

## Retrieve the element with an Odata request

This should fail if the checksum is not MD5

In [ ]:
# Define 2 dates to filter data on
d1 = original_catalog_item["properties"]["published"]
d2 = original_catalog_item["properties"]["published"]

# Endpoint to request
openapi_endpoint = f"/catalog/odata/v1/Products?$filter=PublicationDate gt {d1} and PublicationDate lt {d2}"

# Send request
url = f"{catalog_client.href_service}{openapi_endpoint}"
headers = {'Accept': 'application/json',
          "X-API-Key": os.getenv("RSPY_APIKEY")}

openapi_response = requests.get(url, headers=headers)
odata_items = openapi_response.json()
display(odata_items)

# Find item and check that the UUID is correct
for item in odata_items:
    if item["Name"]==item_id:
        print(item['Checksum'])